In [1]:
# I start by uploading and rading the weather data
import pandas as pd

weather = pd.read_csv('Rogers hourly weather data.csv')


Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


In [2]:
print(weather)
#weather.head()
print(weather.columns)

                                location_id              datetime_beginning  \
0      819ad24f-585c-4050-8b51-9d682a7a7243  2022-01-01 00:00:00.000000 UTC   
1      819ad24f-585c-4050-8b51-9d682a7a7243  2022-01-01 01:00:00.000000 UTC   
2      819ad24f-585c-4050-8b51-9d682a7a7243  2022-01-01 02:00:00.000000 UTC   
3      819ad24f-585c-4050-8b51-9d682a7a7243  2022-01-01 03:00:00.000000 UTC   
4      819ad24f-585c-4050-8b51-9d682a7a7243  2022-01-01 04:00:00.000000 UTC   
...                                     ...                             ...   
17490  819ad24f-585c-4050-8b51-9d682a7a7243  2023-12-30 20:00:00.000000 UTC   
17491  819ad24f-585c-4050-8b51-9d682a7a7243  2023-12-30 21:00:00.000000 UTC   
17492  819ad24f-585c-4050-8b51-9d682a7a7243  2023-12-30 22:00:00.000000 UTC   
17493  819ad24f-585c-4050-8b51-9d682a7a7243  2023-12-30 23:00:00.000000 UTC   
17494  819ad24f-585c-4050-8b51-9d682a7a7243  2023-12-31 00:00:00.000000 UTC   

       temp_f  dewpoint_f  humidity  windspeed_mph 

In [3]:
# I create the base temperature for HDD and CDD. Based on the results of the regression model, I will change the values to improve
# the R^2 value. As discussed in class, it is better to set the baseload to a temperature interval rather than single value.
base_temp_hdd = 35
base_temp_cdd = 45

# I then convert the datetime values in the weather file into datetime objects, and I set it to the index

weather['datetime_beginning'] = pd.to_datetime(weather['datetime_beginning'])
weather.set_index('datetime_beginning', inplace=True)

In [4]:
# I calculate the daily average temperature from the hourly temperature values 
daily_avg_temp = weather['temp_f'].resample('D').mean()
print(daily_avg_temp)

# Then I create a zeros-list for HDD and CDD

hdd = pd.Series(0, index=daily_avg_temp.index)
cdd = pd.Series(0, index=daily_avg_temp.index)

# laslty, I calculate and assign values to HDD or CDD based on the average temperature being smaller or larger than the corresponding base temperature

for day in daily_avg_temp.index:
    avg_temp = daily_avg_temp[day]
    if avg_temp < base_temp_hdd:
        hdd[day] = base_temp_hdd - avg_temp
    elif avg_temp > base_temp_cdd:
        cdd[day] = avg_temp - base_temp_cdd
        
print(hdd)

datetime_beginning
2022-01-01 00:00:00+00:00    52.404167
2022-01-02 00:00:00+00:00    18.646667
2022-01-03 00:00:00+00:00    25.766667
2022-01-04 00:00:00+00:00    36.555833
2022-01-05 00:00:00+00:00    37.512083
                               ...    
2023-12-27 00:00:00+00:00    38.065417
2023-12-28 00:00:00+00:00    34.590833
2023-12-29 00:00:00+00:00    33.813333
2023-12-30 00:00:00+00:00    35.020000
2023-12-31 00:00:00+00:00    47.970000
Freq: D, Name: temp_f, Length: 730, dtype: float64
datetime_beginning
2022-01-01 00:00:00+00:00     0.000000
2022-01-02 00:00:00+00:00    16.353333
2022-01-03 00:00:00+00:00     9.233333
2022-01-04 00:00:00+00:00     0.000000
2022-01-05 00:00:00+00:00     0.000000
                               ...    
2023-12-27 00:00:00+00:00     0.000000
2023-12-28 00:00:00+00:00     0.409167
2023-12-29 00:00:00+00:00     1.186667
2023-12-30 00:00:00+00:00     0.000000
2023-12-31 00:00:00+00:00     0.000000
Freq: D, Length: 730, dtype: float64


In [5]:
#I create a dataframe to align average temperature, HDD and CDD by date
result_df = pd.DataFrame({
    'date': daily_avg_temp.index.date,
    'daily_avg_temp': daily_avg_temp.values,
    'hdd': hdd.values,
    'cdd': cdd.values
})

# I then set the date as the index
result_df.set_index('date', inplace=True)

result_df.index = pd.to_datetime(result_df.index)

result_df

,daily_avg_temp,hdd,cdd
date,,,
2022-01-01,52.404167,0.000000,7.404167
2022-01-02,18.646667,16.353333,0.000000
2022-01-03,25.766667,9.233333,0.000000
2022-01-04,36.555833,0.000000,0.000000
2022-01-05,37.512083,0.000000,0.000000
...,...,...,...
2023-12-27,38.065417,0.000000,0.000000
2023-12-28,34.590833,0.409167,0.000000
2023-12-29,33.813333,1.186667,0.000000


In [6]:
pip install seaborn

Note: you may need to restart the kernel to use updated packages.


In [7]:
# I load the building energy data that I want to weather normalize
file_path = 'Building energy data 2022-2023.xlsx'

energy = pd.read_excel(file_path)

In [8]:
# I convert the Month column in the energy data to datetime, and I make sure its the same object type in result_df
energy['Month'] = pd.to_datetime(energy['Month'])
result_df['Month'] = result_df.index.to_period('M').to_timestamp()

# I Calculate the average monthly temperature in result_df so that I can align it with monthly energy data
monthly_avg_temp = result_df.groupby('Month')['daily_avg_temp'].mean().reset_index()

# I group result_df by Month and sum the HDD and CDD, and get the average temperature, and the I merge everything
monthly_hdd_cdd_temp = result_df.groupby('Month').agg({'hdd': 'sum', 'cdd': 'sum', 'daily_avg_temp': 'mean'}).reset_index()
combined_data = pd.merge(energy, monthly_hdd_cdd_temp, on='Month', how='left')
combined_data.rename(columns={'daily_avg_temp': 'monthly_avg_temp'}, inplace=True)

In [9]:
# I save and display the data 

excel_filename = 'combined_data.xlsx'

combined_data.to_excel(excel_filename)

combined_data

,Month,Energy_kBTU,hdd,cdd,monthly_avg_temp
0,2022-01-01,207587.81,120.232500,16.848333,35.330134
1,2022-02-01,192273.75,105.507083,60.885000,38.241949
2,2022-03-01,142413.91,15.186250,222.702083,49.658589
3,2022-04-01,144601.84,0.000000,376.551667,57.356778
4,2022-05-01,216202.18,0.000000,690.279583,67.267083
5,2022-06-01,207574.12,0.000000,965.627917,77.187597
6,2022-07-01,221767.53,0.000000,1221.828750,84.413831
7,2022-08-01,267570.39,0.000000,1035.028333,78.388011
8,2022-09-01,222621.76,0.000000,802.103750,71.736792
9,2022-10-01,170340.21,0.000000,484.587083,60.262245


In [10]:
# I notice that the enrgy (kBTU) always reaches its mininum when the monthly average temperature is around 50 ºF. 
# That suggests me that the best baseline is around that temperature. I now go back and change the base temperatures for 
# HDD and CDD, and rerun the code until here.

In [11]:
# I build a linear regression model of the monthly energy data regressed on HDD and CDD. This wil confirm if 
# the base temperatures chosen were more or less exact

import statsmodels.api as sm

# I filter the dataset for the baseline year - 2022  
baseline_data = combined_data[combined_data['Month'].dt.year == 2022]

# I define the dependent and independent variables for the baseline year
y_baseline = baseline_data['Energy_kBTU']
X_baseline = baseline_data[['hdd', 'cdd']]

# I add a constant to the independent variable set
X_baseline = sm.add_constant(X_baseline)

# I fit the regression model on the baseline year data
model_baseline = sm.OLS(y_baseline, X_baseline).fit()

print(model_baseline.summary())

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
                            OLS Regression Results                            
Dep. Variable:            Energy_kBTU   R-squared:                       0.755
Model:                            OLS   Adj. R-squared:                  0.700
Method:                 Least Squares   F-statistic:                     13.83
Date:                Thu, 29 Feb 2024   Prob (F-statistic):            0.00180
Time:                        16:18:53   Log-Likelihood:                -134.26
No. Observations:                  12   AIC:                             274.5
Df Residuals:                       9   BIC:                             276.0
Df Model:                           2                                         
Covariance Type:            nonrobust  

/Users/tommy/Downloads/yes/lib/python3.11/site-packages/scipy/stats/_stats_py.py:1971: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=12
  k, _ = kurtosistest(a, axis)


In [12]:
# After some comparison with different values for base temperatures, the highest R^2 value is for base_temp_hdd = 35
# and base_temp_cdd = 45. This shows that the best base temperatures are a little bit lower than the predicted 50 ºF  